# Project Deliverable 1: Data Collection, Cleaning, and Exploration
**Course:** MSCS-634-A01 — Advanced Big Data and Data Mining  
**Student:** Corey Hyde  
**Student ID:** 003044072

## Objective
Explore and prepare the UCI Online Retail dataset for future predictive modeling. All results are generated by running the notebook; conclusions should be updated after reviewing the outputs.

## 1. Dataset selection
The [UCI Online Retail dataset](https://archive.ics.uci.edu/dataset/352/online+retail) contains 541,909 transactions with eight original fields from a UK-based online retailer (2010–2011). Its transaction, product, time, price, and customer fields support future regression, classification, clustering, and association-rule mining. Engineered variables add further attributes. Download **Online Retail.xlsx** from UCI and place it in `data/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_PATH = Path("data/Online Retail.xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download Online Retail.xlsx from https://archive.ics.uci.edu/dataset/352/online+retail and save to data/")
df = pd.read_excel(DATA_PATH, engine="openpyxl")
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)

## 2. Data quality audit
Missing customer identifiers are not fabricated. Duplicate records are removed, and transactions with nonpositive quantity or unit price are excluded from the **sales-only** analytical dataset. Returns and cancellations are kept in a separate dataset for future research. Extremely large purchases are inspected, not automatically deleted.

In [ ]:
print("Missing values:\n", df.isna().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Nonpositive quantity:", (df.Quantity <= 0).sum())
print("Nonpositive price:", (df.UnitPrice <= 0).sum())
print("Cancelled invoices:", df.InvoiceNo.astype(str).str.startswith("C").sum())
print("Missing descriptions:", df.Description.isna().sum())

In [ ]:
clean = df.drop_duplicates().copy()
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"], errors="coerce")
clean["Description"] = clean["Description"].astype("string").str.strip()
clean["Country"] = clean["Country"].astype("string").str.strip()
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str).str.strip()
clean["StockCode"] = clean["StockCode"].astype(str).str.strip()
# Retain non-sale records separately to avoid conflating refunds and purchases.
returns = clean[(clean["Quantity"] <= 0) | clean["InvoiceNo"].str.startswith("C")].copy()
sales = clean.loc[(clean.Quantity > 0) & (clean.UnitPrice > 0) & clean.InvoiceDate.notna() & (~clean.InvoiceNo.str.startswith("C")) & clean.Description.notna() & clean.Country.notna()].copy()
sales["TotalPrice"] = sales.Quantity * sales.UnitPrice
sales["Month"] = sales.InvoiceDate.dt.to_period("M").astype(str)
sales["DayOfWeek"] = sales.InvoiceDate.dt.day_name()
sales["Hour"] = sales.InvoiceDate.dt.hour
print("Raw rows:", len(df), "Sales rows:", len(sales), "Returns/cancellations:", len(returns))
print("Sales attributes:", sales.shape[1])
display(sales.head())

## 3. Explore distributions and outliers
High-value orders can be legitimate bulk purchases. The analysis uses a 99th-percentile display limit only for visualization, without removing those observations from the underlying dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
limit = sales.TotalPrice.quantile(.99)
sns.histplot(sales.loc[sales.TotalPrice <= limit, "TotalPrice"], bins=45, ax=axes[0])
axes[0].set(title="Line-item sales distribution (up to 99th percentile)", xlabel="Line-item revenue (£)")
sns.boxplot(x=sales.loc[sales.TotalPrice <= limit, "TotalPrice"], ax=axes[1])
axes[1].set(title="Line-item sales boxplot (up to 99th percentile)", xlabel="Line-item revenue (£)")
plt.tight_layout(); plt.show()
print("99th percentile:", round(limit, 2))
print("Maximum line-item total:", round(sales.TotalPrice.max(), 2))

In [ ]:
monthly = sales.groupby("Month", sort=True).TotalPrice.sum()
plt.figure(figsize=(12, 4)); monthly.plot(marker="o")
plt.title("Monthly gross sales from positive transactions"); plt.ylabel("Revenue (£)"); plt.xlabel("Month")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("Highest revenue month:", monthly.idxmax())
print("Note: December 2011 is a partial month; avoid direct full-month comparisons.")

In [ ]:
top_countries = sales.groupby("Country").TotalPrice.sum().nlargest(10).sort_values()
plt.figure(figsize=(10, 5)); top_countries.plot(kind="barh")
plt.title("Top 10 countries by gross sales"); plt.xlabel("Revenue (£)"); plt.tight_layout(); plt.show()
print("UK share of positive sales revenue:", f"{sales.loc[sales.Country.eq('United Kingdom'), 'TotalPrice'].sum()/sales.TotalPrice.sum():.1%}")

In [ ]:
numeric = sales[["Quantity", "UnitPrice", "TotalPrice", "Hour"]]
plt.figure(figsize=(7, 5)); sns.heatmap(numeric.corr(method="spearman"), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Spearman correlations among numeric features"); plt.tight_layout(); plt.show()
print("Missing CustomerID in sales:", sales.CustomerID.isna().sum())
print("Distinct invoices:", sales.InvoiceNo.nunique())
print("Distinct products:", sales.StockCode.nunique())

## 4. Prepare data for future deliverables
The full positive-sales dataset supports revenue regression and transaction basket analysis. For customer-level clustering and classification, records with missing CustomerID should be excluded from the customer-level view rather than assigned fabricated identifiers. Future classification targets must be defined carefully to avoid leakage; future-period purchases could support a repeat-purchase label.

In [ ]:
customer_sales = sales.dropna(subset=["CustomerID"]).copy()
customer_features = customer_sales.groupby("CustomerID").agg(
    order_count=("InvoiceNo", "nunique"),
    total_spend=("TotalPrice", "sum"),
    items_purchased=("Quantity", "sum"),
    last_purchase=("InvoiceDate", "max")
).reset_index()
print("Customer-level records:", len(customer_features))
display(customer_features.head())
Path("outputs").mkdir(exist_ok=True)
sales.to_csv("outputs/cleaned_sales.csv", index=False)
returns.to_csv("outputs/returns_and_cancellations.csv", index=False)
customer_features.to_csv("outputs/customer_features.csv", index=False)
print("Saved cleaned datasets in outputs/")

## 5. Interpretation and reflection
**Complete or revise these observations after running the notebook:**

1. **Sales distribution:** Examine the histogram and boxplot. Are most line-item purchases small, with a long right tail? Report the 99th percentile and largest value from the output.
2. **Time patterns:** Identify the highest-revenue month and consider seasonality. December 2011 is incomplete, so comparisons need caution.
3. **Geographic concentration:** Report the United Kingdom share shown above and discuss how country imbalance could affect modeling.
4. **Data quality:** Report missing CustomerID values, duplicates, and removed non-sale rows. Explain why missing customer IDs were not imputed.
5. **Next steps:** For regression, model future sales or basket value using features known at prediction time. For classification, define a time-based repeat-purchase label. For clustering, standardize customer-level spending and frequency measures. For association rules, group product StockCodes by InvoiceNo.

**Limitations:** This is one retailer and one historical period, so findings may not generalize. Positive-sales revenue is not net revenue because returns are excluded. Bulk orders and unusual prices require contextual review.

## References
UCI Machine Learning Repository. (2015). *Online Retail* [Dataset]. https://doi.org/10.24432/C5BW33

Han, J., Pei, J., & Tong, H. (2022). *Data mining* (4th ed.). Elsevier.

Wu, D. (2024). *Data mining with Python*. Taylor & Francis.